# Open VCV -- Training tren Kaggle

### So sanh accelerator (batch=128, img=224, skip-decoder)

| Accelerator | FP TFLOPS | VRAM | Uoc tinh/epoch* | Khuyen nghi |
|---|---|---|---|---|
| RTX 3060 (local) | 25 FP16 | 12GB | ~45 phut | baseline |
| **T4 x2** | 2x65 FP16 | 2x16GB | **~8 phut** | Tot nhat cho GPU |
| T4 x1 | 65 FP16 | 16GB | ~16 phut | Du dung |
| P100 | 21 FP16 | 16GB | ~40 phut | Khong dang doi |
| **TPU v5e-8** | 8x197 BF16 | 128GB | **~3-5 phut** | Tot nhat tong the |

*Uoc tinh tren ImageNet 1.2M anh, q=2, k=0, skip-decoder

**Luu y TPU v5e-8:**
- Lan dau tien chay: **mat 5-15 phut compile graph** (XLA JIT) -- CPU/RAM/TPU hien 0% la binh thuong
- Tu batch thu 2 tro di: chay rat nhanh
- Phai dung `workers=0` de tranh deadlock voi XLA DataLoader
- Khong dung `os.system()` / subprocess -- phai chay inline cung process

### Setup Kaggle:
1. Settings -> Accelerator -> **T4 x2** (GPU) hoac **TPU v5e-8**
2. Add dataset: `imagenet-object-localization-challenge`
3. Internet: **ON**
4. Run All


In [ ]:
# ── Cell 1: Install torch_xla cho TPU v5e ─────────────────────────────────────
# torch_xla đã preinstalled trên Kaggle TPU images,
# nhưng cần đúng version cho v5e-8 (torch_xla 2.x)
import subprocess, sys

result = subprocess.run(['python', '-c', 'import torch_xla; print(torch_xla.__version__)'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'torch_xla already installed: {result.stdout.strip()}')
else:
    print('Installing torch_xla...')
    # Kaggle TPU v5e dùng torch_xla 2.x prebuilt wheel
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch_xla[tpu]',
        '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'
    ])

In [ ]:
# ── Cell 2: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/Bigkatoan/open_vcv.git'
REPO_DIR = '/kaggle/working/open_vcv'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 3: Install Python dependencies ───────────────────────────────────────
os.system('pip install -q -r requirements.txt 2>/dev/null || pip install -q '
          'torch torchvision tqdm matplotlib pandas scikit-learn')

In [ ]:
# -- Cell 4: Kiem tra moi truong ------------------------------------------
# QUAN TRONG: KHONG goi xm.xla_device() o day!
# Goi som se chiem /dev/vfio/0 -> subprocess training se bi 'Device or resource busy'
# TPU se duoc init lan dau khi training bat dau (trong cung process)

import torch
import torch_xla
print(f'torch_xla version : {torch_xla.__version__}')
print(f'torch version     : {torch.__version__}')
print('TPU will be initialized when training starts (do NOT call xm.xla_device() here)')


In [ ]:
# ── Cell 5: Cấu hình paths ────────────────────────────────────────────────────
# ImageNet trên Kaggle dataset
IMAGENET_TRAIN = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'

# Kiểm tra path
if os.path.exists(IMAGENET_TRAIN):
    class_dirs = [d for d in os.listdir(IMAGENET_TRAIN) if os.path.isdir(os.path.join(IMAGENET_TRAIN, d))]
    print(f'ImageNet train: {len(class_dirs)} classes found at {IMAGENET_TRAIN}')
else:
    print(f'WARNING: ImageNet not found at {IMAGENET_TRAIN}')
    print('Hãy add dataset imagenet-object-localization-challenge vào notebook')

In [ ]:
# -- Cell 6: Train tren T4 x2 --------------------------------------------
# Settings -> Accelerator -> T4 x2
# DataParallel: batch=256 duoc chia deu -> 128/GPU
import sys, importlib, os

sys.argv = [
    'main.py',
    '--dataset',        'imagenet',
    '--data-root',      IMAGENET_TRAIN,
    '--device',         'cuda',
    '--epochs',         '100',
    '--batch-size',     '256',
    '--img-size',       '224',
    '--q',              '2',
    '--k',              '0',
    '--workers',        '8',
    '--prefetch',       '4',
    '--skip-decoder',
    '--log-iter-every', '100',
    '--run-name',       'gated_vae_imagenet_t4x2',
]

if '/kaggle/working/open_vcv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/open_vcv')

import main as train_module
importlib.reload(train_module)
train_module.main()


In [ ]:
# ── Cell 7: Benchmark encoders (optional) ─────────────────────────────────────
# So sánh Gated VAE vs các encoders khác trên COCO val
COCO_VAL = '/kaggle/input/coco-2017-dataset/coco2017/val2017'
COCO_ANN = '/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json'

if os.path.exists(COCO_VAL):
    EXP_DIR = 'experiments/gated_vae_imagenet_tpu_v5e'
    os.system(f"""
    python benchmark.py \\
      --exp-dir {EXP_DIR} \\
      --dataset coco \\
      --data-root {COCO_VAL} \\
      --coco-split val \\
      --device xla
    """)
else:
    print('COCO dataset not found, skip benchmark')

In [ ]:
# ── Cell 8: Plot training curves ──────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import glob

EXP_DIRS = sorted(glob.glob('experiments/gated_vae_imagenet_tpu_v5e*'))
if not EXP_DIRS:
    print('No experiment found yet')
else:
    exp = EXP_DIRS[-1]
    losses_csv = os.path.join(exp, 'losses.csv')
    if os.path.exists(losses_csv):
        df = pd.read_csv(losses_csv)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        df.plot('epoch', 'total',  ax=axes[0], title='Total Loss')
        df.plot('epoch', 'union',  ax=axes[1], title='Union Sim')
        df.plot('epoch', 'sparse', ax=axes[2], title='Sparse Loss')
        plt.tight_layout()
        plt.savefig(os.path.join(exp, 'training_curves.png'), dpi=120)
        plt.show()
        print(f'Best epoch: {df.loc[df["total"].idxmin(), "epoch"]}')
    
    # Per-iteration curves
    iter_curves = os.path.join(exp, 'viz', 'curves', 'iter_curves.png')
    if os.path.exists(iter_curves):
        from IPython.display import Image, display
        display(Image(iter_curves))

In [ ]:
# ── Cell 9: Save checkpoint ra Kaggle output ──────────────────────────────────
import shutil, glob

OUTPUT_DIR = '/kaggle/working/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXP_DIRS = sorted(glob.glob('experiments/gated_vae_imagenet_tpu_v5e*'))
if EXP_DIRS:
    exp = EXP_DIRS[-1]
    ckpts = glob.glob(os.path.join(exp, 'checkpoints', '*.pt'))
    for ckpt in ckpts:
        dst = os.path.join(OUTPUT_DIR, os.path.basename(ckpt))
        shutil.copy2(ckpt, dst)
        print(f'Saved: {dst}')
    # Cũng save config và metrics
    for f in ['config.txt', 'losses.csv', 'metrics.csv']:
        src = os.path.join(exp, f)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(OUTPUT_DIR, f))
    print(f'\nAll files saved to {OUTPUT_DIR}')
    print('Kaggle sẽ tự động lưu /kaggle/working/ sau khi notebook kết thúc')